# ARTI 502 Final Project — Member 5: Ensemble & Hyperparameter Experiments

This notebook combines the ResNet18 (Member 3) and DenseNet121 (Member 4)
models for detecting the NIH ChestX-ray14 **Pneumonia** label.

**Rules (to keep the test set clean):**
- Ensemble method, weights and decision threshold are chosen on the **validation** set only.
- The **test** set is scored once, with everything frozen.
- Confidence intervals use a **patient-level** bootstrap, because the splits are patient-level.

**Inputs** (put them in `DATA_DIR`):
- `resnet18_predictions.csv` (Abrar) — `Image Index, Patient ID, split, y_true, y_prob`
- `densenet121_validation_predictions.csv`, `densenet121_test_predictions.csv` (Ghadah) — `Image Index, y_true, p_pneumonia`
- `validation_metadata.csv`, `test_metadata.csv` (E) — patient IDs and the reference labels
- `training_history.csv` from each model (for the training-curve figure)

**Folder layout expected in `DATA_DIR`:** `res/` (unzipped ResNet18 outputs), `dn1/` (unzipped DenseNet121 handoff), `nih/` (unzipped NIH_Dataset_Handoff).

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, confusion_matrix,
                             precision_recall_curve, roc_auc_score, roc_curve)
from sklearn.model_selection import GroupKFold

DATA_DIR = Path("/content/drive/MyDrive/ARTI502/ensemble_inputs")   # folder with res/, dn1/, nih/ subfolders
OUT_DIR = DATA_DIR.parent / "ensemble_outputs"
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
N_BOOT = 2000
COLORS = {"ResNet18": "#2a78d6", "DenseNet121": "#eb6834", "Ensemble": "#1baf7a"}
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"

plt.rcParams.update({
    "font.size": 10, "axes.edgecolor": INK2, "axes.labelcolor": INK, "axes.titlesize": 11,
    "axes.titleweight": "bold", "xtick.color": INK2, "ytick.color": INK2,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.6, "lines.linewidth": 2,
    "legend.frameon": False, "savefig.dpi": 300, "savefig.bbox": "tight",
})

## 1. Load and align the two models' predictions

In [ ]:
res = pd.read_csv(DATA_DIR / "res/resnet18_predictions.csv")
dn = {
    "validation": pd.read_csv(DATA_DIR / "dn1/densenet121_validation_predictions.csv"),
    "test": pd.read_csv(DATA_DIR / "dn1/densenet121_test_predictions.csv"),
}
meta = {s: pd.read_csv(DATA_DIR / f"nih/{s}_metadata.csv") for s in ["validation", "test"]}

data = {}
for split in ["validation", "test"]:
    m = (res[res["split"] == split]
         .merge(dn[split], on="Image Index", suffixes=("_res", "_dn"), validate="1:1")
         .merge(meta[split][["Image Index", "Pneumonia"]], on="Image Index", validate="1:1"))
    assert len(m) == len(meta[split]) == len(dn[split]), f"{split}: row count mismatch"
    assert (m["y_true_res"] == m["y_true_dn"]).all() and (m["y_true_res"] == m["Pneumonia"]).all()
    data[split] = pd.DataFrame({
        "image": m["Image Index"], "patient": m["Patient ID"], "y": m["Pneumonia"].astype(int),
        "p_res": m["y_prob"].clip(1e-6, 1 - 1e-6), "p_dn": m["p_pneumonia"].clip(1e-6, 1 - 1e-6),
    })
    print(f"{split}: {len(m):,} images, {data[split]['y'].sum()} positives, labels agree")

val, test = data["validation"], data["test"]
y_val, y_test = val["y"].values, test["y"].values

## 2. Put the two models on a common scale
The two models output probabilities on different scales (ResNet18 mean ≈ 0.35, DenseNet121
mean ≈ 0.50), so averaging raw probabilities would quietly favour the model with the wider
spread. We convert each probability to a logit and standardise it with the **validation**
mean and SD, then reuse those same constants on test.

In [ ]:
def logit(p):
    return np.log(p / (1 - p))

scale = {}
for k in ["res", "dn"]:
    lv = logit(val[f"p_{k}"].values)
    scale[k] = (lv.mean(), lv.std())
for df in (val, test):
    for k in ["res", "dn"]:
        df[f"z_{k}"] = (logit(df[f"p_{k}"].values) - scale[k][0]) / scale[k][1]

corr = {s: float(pd.Series(d["p_res"]).corr(pd.Series(d["p_dn"]), method="spearman"))
        for s, d in data.items()}
print("Spearman correlation between models:", {k: round(v, 3) for k, v in corr.items()})

## 3. Ensemble experiments (validation set only)
Candidate combination rules, each scored on validation AP (the team's selection metric) and AUROC:
1. **Probability average** — mean of the two sigmoid outputs
2. **Standardised-logit average** — mean of the z-scored logits
3. **Maximum** — the higher of the two z-scores
4. **Weighted standardised-logit average** — weight *w* on ResNet18, swept 0→1
5. **Stacking** — logistic regression on the two z-scores (scored with patient-grouped 5-fold CV)

In [ ]:
def score(y, s):
    return {"AP": average_precision_score(y, s), "AUROC": roc_auc_score(y, s)}

def ens(df, w=0.5, how="zavg"):
    if how == "pavg":
        return w * df["p_res"].values + (1 - w) * df["p_dn"].values
    if how == "max":
        return np.maximum(df["z_res"].values, df["z_dn"].values)
    return w * df["z_res"].values + (1 - w) * df["z_dn"].values

# weight sweep
weights = np.round(np.arange(0, 1.0001, 0.05), 2)
sweep = pd.DataFrame([{"w_resnet18": w, **{f"val_{k}": v for k, v in score(y_val, ens(val, w)).items()}}
                      for w in weights])

# stacking with patient-grouped CV on validation
X_val = val[["z_res", "z_dn"]].values
oof = np.zeros(len(val))
for tr, te in GroupKFold(n_splits=5).split(X_val, y_val, groups=val["patient"]):
    lr = LogisticRegression().fit(X_val[tr], y_val[tr])
    oof[te] = lr.decision_function(X_val[te])
stack = LogisticRegression().fit(X_val, y_val)

w_best = float(sweep.loc[sweep["val_AP"].idxmax(), "w_resnet18"])
methods = pd.DataFrame([
    {"method": "ResNet18 alone", **score(y_val, val["z_res"])},
    {"method": "DenseNet121 alone", **score(y_val, val["z_dn"])},
    {"method": "Probability average (equal)", **score(y_val, ens(val, 0.5, "pavg"))},
    {"method": "Standardised-logit average (equal)", **score(y_val, ens(val, 0.5))},
    {"method": "Maximum of z-scores", **score(y_val, ens(val, how="max"))},
    {"method": f"Weighted z-average (best w={w_best:.2f}, in-sample)", **score(y_val, ens(val, w_best))},
    {"method": "Stacking, logistic regression (5-fold grouped CV)", **score(y_val, oof)},
])
print(methods.round(4).to_string(index=False))

**Choosing the weight.** The weight with the highest validation AP is picked on only 208
positive images, so it is noisy. To check whether that weight is a real optimum or just
noise, we bootstrap patients in the validation set and record which weight wins each time.

In [ ]:
rng = np.random.default_rng(SEED)
groups_val = list(val.groupby("patient").indices.values())
win_counts = pd.Series(0, index=weights)
Z = np.stack([ens(val, w) for w in weights])          # (n_weights, n_images)
for _ in range(300):
    idx = np.concatenate([groups_val[i] for i in rng.integers(0, len(groups_val), len(groups_val))])
    yy = y_val[idx]
    if yy.sum() == 0:
        continue
    aps = [average_precision_score(yy, Z[j, idx]) for j in range(len(weights))]
    win_counts[weights[int(np.argmax(aps))]] += 1
print("How often each ResNet18 weight wins (validation bootstrap):")
print(win_counts[win_counts > 0].to_string())

In [ ]:
# Final choice rule: keep equal weights unless the AP-best weight (a) wins most bootstrap
# resamples AND (b) beats equal weights on both validation AP and AUROC.
# Reason: both models already used this validation set to pick their checkpoint (ResNet18 from
# 8 epochs, DenseNet121 from 3), so their validation scores are optimistic — ResNet18's more so —
# and a weight tuned on the same 208 positives would inherit that bias.
eq = score(y_val, ens(val, 0.5))
best = score(y_val, ens(val, w_best))
win_share = win_counts[w_best] / win_counts.sum()
stable = win_share > 0.5 and best["AP"] > eq["AP"] and best["AUROC"] > eq["AUROC"]
FINAL_W = w_best if stable else 0.5
print(f"AP-best w={w_best}: AP {best['AP']:.4f}, AUROC {best['AUROC']:.4f}, wins {win_share:.0%} of resamples")
print(f"Equal w=0.5:    AP {eq['AP']:.4f}, AUROC {eq['AUROC']:.4f}")
print(f"Chosen ResNet18 weight: {FINAL_W}")

val["ens"] = ens(val, FINAL_W)
test["ens"] = ens(test, FINAL_W)

## 4. Decision threshold (validation only)
Same rule as Members 3 and 4: the threshold that maximises validation F1. We also report a
**high-sensitivity** operating point (validation recall ≥ 0.80), which is closer to how a
screening tool would be used.

In [ ]:
def f1_threshold(y, s):
    p, r, t = precision_recall_curve(y, s)
    f1 = 2 * p[:-1] * r[:-1] / np.maximum(p[:-1] + r[:-1], 1e-12)
    return float(t[np.argmax(f1)])

def recall_threshold(y, s, target=0.80):
    fpr, tpr, t = roc_curve(y, s)
    return float(t[np.argmax(tpr >= target)])

score_cols = {"ResNet18": "p_res", "DenseNet121": "p_dn", "Ensemble": "ens"}
# Sensitivity analysis only (not the reported model): the AP-best weight
val["ens_w_best"], test["ens_w_best"] = ens(val, w_best), ens(test, w_best)
thr_f1 = {"ResNet18": 0.7549149990081787, "DenseNet121": 0.7319465279579163}   # teammates' thresholds
thr_f1["Ensemble"] = f1_threshold(y_val, val["ens"])
thr_sens = {m: recall_threshold(y_val, val[c]) for m, c in score_cols.items()}
for m, c in score_cols.items():          # sanity check that the teammates' thresholds reproduce
    if m != "Ensemble":
        assert abs(f1_threshold(y_val, val[c]) - thr_f1[m]) < 1e-6, m
print("Max-F1 thresholds:", {k: round(v, 4) for k, v in thr_f1.items()})

## 5. Test-set evaluation (run once, everything frozen)

In [ ]:
def metrics_at(y, s, thr):
    pred = (s >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn)
    return {"AUROC": roc_auc_score(y, s), "AP": average_precision_score(y, s),
            "Accuracy": (tp + tn) / len(y), "Precision": prec, "Recall (sensitivity)": rec,
            "Specificity": tn / (tn + fp), "F1": 2 * prec * rec / (prec + rec) if prec + rec else 0.0,
            "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn), "Threshold": thr}

rows = []
for split, df in [("Validation", val), ("Test", test)]:
    for m, c in score_cols.items():
        rows.append({"split": split, "model": m, "operating_point": "max-F1",
                     **metrics_at(df["y"].values, df[c].values, thr_f1[m])})
        rows.append({"split": split, "model": m, "operating_point": "recall>=0.80 on val",
                     **metrics_at(df["y"].values, df[c].values, thr_sens[m])})
t_wb = f1_threshold(y_val, val["ens_w_best"])
for split, df in [("Validation", val), ("Test", test)]:
    rows.append({"split": split, "model": f"Ensemble w={w_best:.2f} (sensitivity analysis)",
                 "operating_point": "max-F1", **metrics_at(df["y"].values, df["ens_w_best"].values, t_wb)})
results = pd.DataFrame(rows)
results.to_csv(OUT_DIR / "all_results_val_and_test.csv", index=False)
main = results[(results.split == "Test") & (results.operating_point == "max-F1")].set_index("model")
print(results[results.model.str.contains("sensitivity")][["split", "AUROC", "AP", "F1"]].round(4))
main = main.loc[list(score_cols)]
print(main.drop(columns=["split", "operating_point"]).round(4).T.to_string())

## 6. Patient-level bootstrap: confidence intervals and paired differences

In [ ]:
rng = np.random.default_rng(SEED)
groups = list(test.groupby("patient").indices.values())
S = {m: test[c].values for m, c in score_cols.items()}
boot = []
for _ in range(N_BOOT):
    idx = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
    yy = y_test[idx]
    if yy.sum() == 0:
        continue
    row = {}
    for m in S:
        row[f"{m}_AUROC"] = roc_auc_score(yy, S[m][idx])
        row[f"{m}_AP"] = average_precision_score(yy, S[m][idx])
    boot.append(row)
boot = pd.DataFrame(boot)

ci_rows = []
for m in S:
    for k in ["AUROC", "AP"]:
        b = boot[f"{m}_{k}"]
        ci_rows.append({"model": m, "metric": k, "point": main.loc[m, k],
                        "ci_low": b.quantile(0.025), "ci_high": b.quantile(0.975)})
ci = pd.DataFrame(ci_rows)

diff_rows = []
for other in ["ResNet18", "DenseNet121"]:
    for k in ["AUROC", "AP"]:
        d = boot[f"Ensemble_{k}"] - boot[f"{other}_{k}"]
        diff_rows.append({"comparison": f"Ensemble − {other}", "metric": k,
                          "difference": main.loc["Ensemble", k] - main.loc[other, k],
                          "ci_low": d.quantile(0.025), "ci_high": d.quantile(0.975),
                          "share_of_resamples_ensemble_better": (d > 0).mean()})
diffs = pd.DataFrame(diff_rows)
ci.to_csv(OUT_DIR / "test_bootstrap_ci.csv", index=False)
diffs.to_csv(OUT_DIR / "test_paired_differences.csv", index=False)
print(ci.round(4).to_string(index=False)); print(diffs.round(4).to_string(index=False))

## 7. Report tables

In [ ]:
def fmt_ci(m, k):
    r = ci[(ci.model == m) & (ci.metric == k)].iloc[0]
    return f"{r.point:.3f} ({r.ci_low:.3f}–{r.ci_high:.3f})"

table_main = pd.DataFrame({
    m: {"AUROC (95% CI)": fmt_ci(m, "AUROC"), "AP (95% CI)": fmt_ci(m, "AP"),
        "Accuracy": f"{main.loc[m, 'Accuracy']:.3f}", "Precision": f"{main.loc[m, 'Precision']:.3f}",
        "Recall (sensitivity)": f"{main.loc[m, 'Recall (sensitivity)']:.3f}",
        "Specificity": f"{main.loc[m, 'Specificity']:.3f}", "F1": f"{main.loc[m, 'F1']:.3f}",
        "Threshold (from validation)": f"{main.loc[m, 'Threshold']:.3f}"}
    for m in S})
table_main.index.name = "Metric (test set, n=16,374, 193 positive)"
table_main.to_csv(OUT_DIR / "table_main_comparison.csv")

sens = results[(results.split == "Test") & (results.operating_point != "max-F1")].set_index("model")
print(sens[["Recall (sensitivity)", "Specificity", "Precision", "TP", "FP", "FN", "TN", "Threshold"]].round(3))
table_sens = sens[["Recall (sensitivity)", "Specificity", "Precision", "TP", "FP", "FN", "TN"]].round(3)
table_sens.to_csv(OUT_DIR / "table_high_sensitivity.csv")

table_params = pd.DataFrame([
    ["ResNet18", "Learning rate", "1e-4 (AdamW, cosine schedule)"],
    ["ResNet18", "Weight decay", "1e-4"], ["ResNet18", "Batch size", "64"],
    ["ResNet18", "Epochs / early stopping", "15 max, patience 4 → stopped at 8, best epoch 4"],
    ["ResNet18", "Class imbalance", "pos_weight = 81.9"],
    ["ResNet18", "Augmentation", "Horizontal flip (p = 0.5)"],
    ["ResNet18", "Decision threshold", f"{thr_f1['ResNet18']:.4f} (max validation F1)"],
    ["DenseNet121", "Learning rate", "3e-5 (AdamW) — chosen over 1e-4"],
    ["DenseNet121", "Weight decay", "0.01"], ["DenseNet121", "Batch size", "16"],
    ["DenseNet121", "Epochs", "3, best epoch 1"],
    ["DenseNet121", "Class imbalance", "pos_weight = 81.9"],
    ["DenseNet121", "Augmentation", "Horizontal flip (p = 0.5)"],
    ["DenseNet121", "Decision threshold", f"{thr_f1['DenseNet121']:.4f} (max validation F1)"],
    ["Ensemble", "Combination rule", "Weighted average of standardised logits"],
    ["Ensemble", "ResNet18 weight w", f"{FINAL_W:.2f} (DenseNet121 = {1 - FINAL_W:.2f})"],
    ["Ensemble", "Standardisation", "Validation mean/SD of each model's logit"],
    ["Ensemble", "Decision threshold", f"{thr_f1['Ensemble']:.4f} (max validation F1)"],
], columns=["Model", "Parameter", "Optimal value chosen"])
table_params.to_csv(OUT_DIR / "table_optimal_parameters.csv", index=False)
methods.to_csv(OUT_DIR / "table_ensemble_methods_validation.csv", index=False)
sweep.to_csv(OUT_DIR / "ensemble_weight_sweep_validation.csv", index=False)
print(table_main.to_string())

## 8. Figures

In [ ]:
def save(fig, name):
    fig.savefig(FIG_DIR / name)
    plt.close(fig)

# Fig A — ensemble weight sweep (hyperparameter search plot for Section 5.4)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for a, k in zip(ax, ["AP", "AUROC"]):
    a.plot(sweep["w_resnet18"], sweep[f"val_{k}"], color=COLORS["Ensemble"], marker="o", ms=4)
    a.axvline(FINAL_W, color=INK2, ls="--", lw=1)
    a.annotate(f"chosen w = {FINAL_W:.2f}", (FINAL_W, a.get_ylim()[0]), xytext=(4, 6),
               textcoords="offset points", color=INK2, fontsize=9)
    a.set(xlabel="Weight on ResNet18 (1 − w on DenseNet121)", ylabel=f"Validation {k}",
          title=f"Ensemble weight search — validation {k}")
    a.text(0, sweep[f"val_{k}"].iloc[0], " DenseNet121\n only", fontsize=8, color=INK2, va="bottom")
    a.text(1, sweep[f"val_{k}"].iloc[-1], "ResNet18\nonly ", fontsize=8, color=INK2, va="bottom", ha="right")
save(fig, "fig_ensemble_weight_search.png")

# Fig B — ROC curves, test
fig, ax = plt.subplots(figsize=(5, 4.6))
for m in S:
    fpr, tpr, _ = roc_curve(y_test, S[m])
    ax.plot(fpr, tpr, color=COLORS[m], label=f"{m} (AUROC {main.loc[m, 'AUROC']:.3f})")
ax.plot([0, 1], [0, 1], color=INK2, ls=":", lw=1, label="Chance")
ax.set(xlabel="False positive rate (1 − specificity)", ylabel="True positive rate (sensitivity)",
       title="ROC curves — test set", xlim=(0, 1), ylim=(0, 1.01))
ax.legend(loc="lower right")
save(fig, "fig_roc_test.png")

# Fig C — precision-recall curves, test
fig, ax = plt.subplots(figsize=(5, 4.6))
for m in S:
    p, r, _ = precision_recall_curve(y_test, S[m])
    ax.plot(r, p, color=COLORS[m], label=f"{m} (AP {main.loc[m, 'AP']:.3f})")
ax.axhline(y_test.mean(), color=INK2, ls=":", lw=1, label=f"No skill (prevalence {y_test.mean():.3f})")
ax.set(xlabel="Recall (sensitivity)", ylabel="Precision", title="Precision–recall curves — test set",
       xlim=(0, 1), ylim=(0, 0.12))
ax.legend(loc="upper right")
save(fig, "fig_pr_test.png")

# Fig D — confusion matrices, test, max-F1 thresholds
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)
fig.subplots_adjust(wspace=0.08)
for n, (a, m) in enumerate(zip(axes, S)):
    cm = confusion_matrix(y_test, (S[m] >= thr_f1[m]).astype(int), labels=[0, 1])
    a.imshow(np.log10(cm + 1), cmap="Blues", vmin=0, vmax=5)
    for i in range(2):
        for j in range(2):
            a.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", fontsize=12,
                   color="white" if cm[i, j] > 1000 else INK)
    a.set(xticks=[0, 1], yticks=[0, 1], xticklabels=["No pneumonia", "Pneumonia"],
          yticklabels=["No pneumonia", "Pneumonia"], xlabel="Predicted",
          ylabel="Actual" if n == 0 else "", title=f"{m} (threshold {thr_f1[m]:.3f})")
    a.grid(False)
fig.suptitle("Confusion matrices — test set, max-F1 thresholds chosen on validation", fontsize=11, y=1.05)
save(fig, "fig_confusion_matrices_test.png")

# Fig E — ensemble method comparison on validation
fig, ax = plt.subplots(figsize=(8, 3.6))
mm = methods.iloc[::-1]
bar_colors = [COLORS["ResNet18"] if "ResNet18 alone" in s else COLORS["DenseNet121"] if "DenseNet121 alone" in s
              else COLORS["Ensemble"] for s in mm["method"]]
ax.barh(mm["method"], mm["AP"], color=bar_colors, height=0.6)
for yv, v in enumerate(mm["AP"]):
    ax.text(v, yv, f" {v:.4f}", va="center", fontsize=9, color=INK)
ax.axvline(y_val.mean(), color=INK2, ls=":", lw=1)
ax.text(y_val.mean(), len(mm) - 0.45, " no-skill AP (prevalence)", fontsize=8, color=INK2)
ax.set(xlabel="Validation average precision (AP)", title="Ensemble combination rules — validation set",
       xlim=(0, mm["AP"].max() * 1.18))
ax.grid(axis="y", visible=False)
save(fig, "fig_ensemble_methods_validation.png")

# Fig F — training curves for both CNNs (no shared y-scale problems: one panel per model)
h_res = pd.read_csv(DATA_DIR / "res/training_history.csv")
h_dn = pd.read_csv(DATA_DIR / "dn1/training_history.csv")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(h_res["epoch"], h_res["val_ap"], color=COLORS["ResNet18"], marker="o", label="ResNet18")
ax[0].plot(h_dn["epoch"], h_dn["val_ap"], color=COLORS["DenseNet121"], marker="o", label="DenseNet121")
ax[0].axhline(y_val.mean(), color=INK2, ls=":", lw=1, label="No skill")
ax[0].set(xlabel="Epoch", ylabel="Validation AP", title="Validation AP by epoch", ylim=(0, 0.045))
ax[0].legend()
ax[1].plot(h_res["epoch"], h_res["train_loss"], color=COLORS["ResNet18"], marker="o", label="ResNet18 train")
ax[1].plot(h_dn["epoch"], h_dn["train_loss"], color=COLORS["DenseNet121"], marker="o", label="DenseNet121 train")
ax[1].plot(h_dn["epoch"], h_dn["val_loss"], color=COLORS["DenseNet121"], marker="s", ls="--",
           label="DenseNet121 validation")
ax[1].set(xlabel="Epoch", ylabel="Weighted BCE loss", title="Loss by epoch")
ax[1].legend()
save(fig, "fig_training_curves_both.png")

# Fig G — how much the models agree (diversity)
fig, ax = plt.subplots(figsize=(5, 4.6))
neg, pos = test[test.y == 0], test[test.y == 1]
ax.scatter(neg["z_res"], neg["z_dn"], s=3, color="#b9b8b2", alpha=0.35, label="No pneumonia", rasterized=True)
ax.scatter(pos["z_res"], pos["z_dn"], s=14, color=COLORS["Ensemble"], edgecolor="white", lw=0.5, label="Pneumonia")
ax.set(xlabel="ResNet18 score (standardised logit)", ylabel="DenseNet121 score (standardised logit)",
       title=f"Model agreement — test (Spearman ρ = {corr['test']:.2f})")
ax.legend(loc="upper left", markerscale=2)
save(fig, "fig_model_agreement_test.png")

summary = {"chosen_resnet18_weight": FINAL_W, "best_weight_in_sample": w_best,
           "standardisation_constants": {k: list(map(float, v)) for k, v in scale.items()},
           "thresholds_max_f1": thr_f1, "thresholds_recall80": thr_sens,
           "spearman_between_models": corr, "n_bootstrap": N_BOOT, "seed": SEED}
(OUT_DIR / "ensemble_settings.json").write_text(json.dumps(summary, indent=2))
print("Saved:", sorted(p.name for p in OUT_DIR.rglob("*") if p.is_file()))